# Distributed LLM Fine-Tuning with Ray Train and CodeFlare SDK

This notebook demonstrates the **complete end-to-end workflow** for distributed LLM fine-tuning using Ray Train on OpenShift AI.

## Prerequisites

- OpenShift AI cluster with GPU nodes (4x NVIDIA L40S recommended)
- DataScience Project with Workbench created and the directory /ray uploaded
- kubectl configured with access to your namespace

## Workflow Steps

1. **Create Ray Cluster** - Deploy Ray cluster with 4 GPUs using CodeFlare SDK
2. **Submit Training Job** - Launch distributed fine-tuning job
3. **Monitor Training** - Check job status and view metrics
4. **Cleanup** - Delete cluster resources


## Key Features

✅ **No Docker Build Required** - Uses CodeFlare MODH image with Ray pre-installed  
✅ **Fast Iteration** - Update code and redeploy in 30-60 seconds  
✅ **Programmatic Deployment** - Python API instead of YAML files  
✅ **Rich Monitoring** - Ray Dashboard with real-time metrics  

---

## Step 1: Setup and Installation

First, let's install and import the CodeFlare SDK.

**Note:** This notebook requires CodeFlare SDK >= 0.32.0


In [ ]:
# CodeFlare SDK imports (version 0.32.0)
import codeflare_sdk
from codeflare_sdk import Cluster, ClusterConfiguration
from codeflare_sdk import RayJob

print("✓ CodeFlare SDK imported successfully")
print(f"  Version: {codeflare_sdk.__version__}")
print("  - Cluster, ClusterConfiguration")
print("  - RayJob (for RayJob CR submission)")
print("")
print("ℹ️  This notebook requires CodeFlare SDK >= 0.32.0")
print("   Upgrade if needed: %pip install --upgrade codeflare-sdk")


In [ ]:
# Cluster configuration
CLUSTER_NAME = "smollm3-ray-cluster"
NAMESPACE = "<your-datascience-project-namespace>"  # Change to your OpenShift namespace
NUM_WORKERS = 3  # 3 workers + 1 head = 4 total GPUs

# Resource configuration (per node)
HEAD_CPUS = 6
HEAD_MEMORY = 20  # GB
HEAD_GPUS = 1

WORKER_CPUS = 6
WORKER_MEMORY = 20  # GB
WORKER_GPUS = 1

# Training parameters
EPOCHS = 4
BATCH_SIZE = 8
LEARNING_RATE = "5e-5"
OUTPUT_DIR = "/tmp/models"

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Cluster: {CLUSTER_NAME}")
print(f"Namespace: {NAMESPACE}")
print(f"Workers: {NUM_WORKERS} (+ 1 head = {NUM_WORKERS + 1} total GPUs)")
print(f"\nTraining Parameters:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Output dir: {OUTPUT_DIR}")
print("=" * 80)


---

## Step 2: Create Ray Cluster

Now let's create a Ray cluster using CodeFlare SDK. This will:
- Create a RayCluster resource
- Spin up head and worker pods with GPUs
- Configure services for Ray Dashboard
- Set up all networking automatically

**Note:** CodeFlare SDK handles ALL Kubernetes resource creation - no manual YAML needed!


In [ ]:
# Create ClusterConfiguration
cluster_config = ClusterConfiguration(
    name=CLUSTER_NAME,
    namespace=NAMESPACE,
    num_workers=NUM_WORKERS,
    head_cpu_requests=HEAD_CPUS,
    head_memory_requests=HEAD_MEMORY,
    worker_cpu_requests=WORKER_CPUS,
    worker_memory_requests=WORKER_MEMORY,
    head_cpu_limits=HEAD_CPUS,
    head_memory_limits=HEAD_MEMORY,
    worker_cpu_limits=WORKER_CPUS,
    worker_memory_limits=WORKER_MEMORY,
    write_to_file=False,  # Set to True to generate YAML for reference
    # GPU configuration
    head_extended_resource_requests={"nvidia.com/gpu": HEAD_GPUS},
    worker_extended_resource_requests={"nvidia.com/gpu": WORKER_GPUS},
)

print("✓ ClusterConfiguration created")
print(f"Total resources: {HEAD_CPUS + NUM_WORKERS*WORKER_CPUS} CPU, "
      f"{HEAD_MEMORY + NUM_WORKERS*WORKER_MEMORY}GB RAM, "
      f"{HEAD_GPUS + NUM_WORKERS*WORKER_GPUS} GPU")

In [ ]:
# Create and deploy the cluster
print("=" * 80)
print("DEPLOYING RAY CLUSTER")
print("=" * 80)
print("Creating cluster resources...")
print("  - RayCluster custom resource")
print("  - Head pod (1x)")
print(f"  - Worker pods ({NUM_WORKERS}x)")
print("  - Services (head service, dashboard)")
print("")

cluster = Cluster(cluster_config)
cluster.apply()

print("✓ Cluster deployment initiated")
print("  Waiting for cluster to be ready (timeout: 600s)...")

try:
    cluster.wait_ready(timeout=600)
    print("\n✓ CLUSTER IS READY!")
    print("=" * 80)
except Exception as e:
    print(f"\n✗ Cluster failed to become ready: {e}")
    print("\nChecking cluster status using CodeFlare SDK:")
    try:
        print(f"  Status: {cluster.status()}")
        print("\nCluster details:")
        print(cluster.details())
    except Exception as detail_error:
        print(f"  Could not retrieve cluster details: {detail_error}")
    
    print("\nAlternative: Check manually with kubectl:")
    print(f"  kubectl get raycluster {CLUSTER_NAME} -n {NAMESPACE}")
    print(f"  kubectl get pods -l ray.io/cluster={CLUSTER_NAME} -n {NAMESPACE}")
    raise


In [ ]:
# Check cluster status and details using CodeFlare SDK
print("=" * 80)
print("CLUSTER STATUS CHECK")
print("=" * 80)

# Get cluster status
status = cluster.status()
print(f"\nCluster Status: {status}")

# Get detailed cluster information
print("\n" + "=" * 80)
print("CLUSTER DETAILS")
print("=" * 80)
details = cluster.details()
print(details)

# List all workers
print("\n" + "=" * 80)
print("CLUSTER WORKERS")
print("=" * 80)
print(f"Expected workers: {NUM_WORKERS}")
print(f"Head node: 1")
print(f"Total GPUs: {NUM_WORKERS + 1}")

print("\n✓ Cluster is fully operational and ready for job submission!")
print("=" * 80)


## Step 3: Submit Training Job

Now let's submit our distributed fine-tuning job to the Ray cluster.

The job will:
1. **Upload code** - Current directory including `ray_training.py`
2. **Install packages** - TRL, transformers, etc. dynamically
3. **Execute training** - Distributed across all 4 GPUs
4. **Save results** - Checkpoints and final model


In [ ]:
# Define runtime environment
runtime_env = {
    "working_dir": "./scripts",  # Upload current directory to Ray cluster
    "pip": ["-r", "requirements.txt"],
    "env_vars": {
        "OUTPUT_DIR": str(OUTPUT_DIR),
        "NUM_WORKERS": str(NUM_WORKERS + 1),  # head + workers
    }
}

print("=" * 80)
print("RUNTIME ENVIRONMENT")
print("=" * 80)
print(f"Working directory: {runtime_env['working_dir']}")
print(f"\nPackages to install:")
for pkg in runtime_env['pip']:
    print(f"  - {pkg}")
print(f"\nEnvironment variables:")
for key, value in runtime_env['env_vars'].items():
    print(f"  {key}={value}")
print("=" * 80)


In [ ]:
# Submit training job as RayJob Custom Resource
print("=" * 80)
print("SUBMITTING RAYJOB CR")
print("=" * 80)

# Define job parameters
entrypoint = "python ray_training.py"
job_name = f"{CLUSTER_NAME}-training-job"

print(f"Job name: {job_name}")
print(f"Entrypoint: {entrypoint}")
print("")


rayjob = RayJob(
    job_name=job_name,
    cluster_name=CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint=entrypoint,
    runtime_env=runtime_env,
)

rayjob.submit()

print("")
print("=" * 80)
print("✅ RAYJOB SUBMITTED SUCCESSFULLY!")
print("=" * 80))
print(f"  Job Name: {job_name}")
print("")
print("A RayJob resource has been created.")
print("")
print("Check RayJob resource:")
print(f"  kubectl get rayjob {job_name} -n {NAMESPACE}")
print(f"  kubectl describe rayjob {job_name} -n {NAMESPACE}")
print("")
print("Proceed to the next cell to monitor progress.")
print("=" * 80)



### View Job Logs

Let's retrieve the job logs to see training progress:


### Access Ray Dashboard

Let's get the Ray Dashboard URL using CodeFlare SDK:

In [ ]:
# Get Ray Dashboard URL using CodeFlare SDK
print("=" * 80)
print("RAY DASHBOARD ACCESS")
print("=" * 80)

# Get dashboard URL from cluster
dashboard_url = cluster.cluster_dashboard_uri()
print(f"\nRay Dashboard URL: {dashboard_url}")

### Cleanup

Now, let's cleanup the cluster when the job is completed

In [ ]:
# Check current cluster and job resources using CodeFlare SDK
print("=" * 80)
print("CURRENT RESOURCES")
print("=" * 80)

# Check cluster status
print("\nCluster Status:")
print(f"  {cluster.status()}")

print("\nCluster Details:")
try:
    print(cluster.details())
except Exception as e:
    print(f"  Error: {e}")


In [ ]:
# Delete the cluster
print("=" * 80)
print("DELETING RAY CLUSTER")
print("=" * 80)
print(f"Cluster: {CLUSTER_NAME}")
print(f"Namespace: {NAMESPACE}")
print("")

try:
    cluster.down()
    print("✓ Cluster deletion initiated")
    print("  Resources are being cleaned up...")
    
    # Wait for resources to be deleted
    time.sleep(10)
    
    print("\n✓ CLEANUP COMPLETE!")
    print("=" * 80)
    
except Exception as e:
    print(f"✗ Error during cleanup: {e}")
    print("\nManual cleanup commands:")
    print(f"  kubectl delete raycluster {CLUSTER_NAME} -n {NAMESPACE}")
    print(f"  kubectl delete pods -l ray.io/cluster={CLUSTER_NAME} -n {NAMESPACE} --force")


In [ ]:
# Verify cleanup
print("=" * 80)
print("VERIFYING CLEANUP")
print("=" * 80)
print("")
print("Remaining RayClusters:")
!kubectl get raycluster -n {NAMESPACE}
print("")
print("Remaining Ray pods:")
!kubectl get pods -l ray.io/cluster={CLUSTER_NAME} -n {NAMESPACE}
print("")
print("✓ If no resources are listed above, cleanup was successful!")
print("=" * 80)


---

## 🎉 Summary

You've successfully completed the distributed LLM fine-tuning workflow!

### What You Did:

✅ **Created Ray Cluster** - 4 GPUs (1 head + 3 workers)  
✅ **Submitted Training Job** - Distributed fine-tuning across all GPUs  
✅ **Monitored Progress** - Real-time job status  
✅ **Cleaned Up Resources** - Deleted cluster and jobs  

### Key Achievements:

- **No Docker Build** - Used CodeFlare MODH image directly
- **Fast Iteration** - Code uploaded via `runtime_env`
- **Version Compatible** - Works with any CodeFlare SDK version
- **Programmatic** - All operations via Python (no YAML!)

---

**Happy Training! 🚀**
